# Impulse — KeyValueStore Solver on a Reduced Data Model

This demo drives the **`KeyValueStoreSolver`** against a *reduced* silver-layer
model that has **no `container_tags` and no `channel_tags`** tables. Instead:

- Channel identity (`channel_name`, `unit`) lives as **columns on `channel_metrics`**.
- Logical channels are resolved through a **`channel_mapping`** alias table.
- Values are rescaled automatically through a **`unit_conversion`** table when an
  aliased channel's source unit differs from its target unit.

Container 1 carries the vehicle-speed signal **twice** — as `Vehicle Speed Sensor`
(km/h) and as `veh_spd_ms` (m/s). Both map to the `vehicle_speed` alias, so the demo
also shows how a **priority** tie-break picks a single physical channel.

Run the cells top to bottom on a serverless Databricks cluster — fill in the
**Catalog**, **Schema**, and **Table Prefix** widgets that appear after running cell 1.

Full walkthrough at the [KeyValueStore Solver page](https://databrickslabs.github.io/impulse/docs/tutorial/key_value_store).

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "", "Table Prefix")

In [ ]:
import sys, os
import pandas as pd

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")

if not CATALOG or not SCHEMA or not TABLE_PREFIX:
    raise ValueError("Please set Catalog, Schema, and Table Prefix widgets above before running.")

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
DEMOS_DIR = "/Workspace" + "/".join(nb_path.split("/")[:-1])
REPO_ROOT = "/Workspace" + "/".join(nb_path.split("/")[:-2])
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# The reduced model deliberately omits container_tags and channel_tags.
csv_dir = os.path.join(DEMOS_DIR, "data", "reporting")
for t in ["container_metrics", "channel_metrics", "channels", "channel_mapping", "unit_conversion"]:
    (spark.createDataFrame(pd.read_csv(f"{csv_dir}/{t}.csv"))
          .write.mode("overwrite")
          .saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_{t}"))
print(f"Loaded 5 silver-layer tables (no container_tags / channel_tags) under {CATALOG}.{SCHEMA}.{TABLE_PREFIX}_*")

## Configure the report

Two things differ from the `DeltaSolver` getting-started demo:

- `source` lists **no** `container_tags_table` / `channel_tags_table`, but **does**
  list `channel_mapping_table` and `unit_conversion_table`.
- `solver_config.channel_mapping.join_keys` overrides the default alias join to use
  only `source_channel ↔ channel_name`, so no `data_key` column is needed.

In [ ]:
from databricks.sdk import WorkspaceClient

from impulse_reporting.aggregations.histogram import HistogramDuration
from impulse_reporting.core.page import Page
from impulse_reporting.core.report import Report
from impulse_reporting.events.basic_event import BasicEvent

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"

config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "channel_metrics_table":   f"{pfx}_channel_metrics",
        "channels_uri":            f"{pfx}_channels",
        "channel_mapping_table":   f"{pfx}_channel_mapping",
        "unit_conversion_table":   f"{pfx}_unit_conversion",
        # NOTE: no container_tags_table, no channel_tags_table
    },
    "unity_sink": {
        "catalog": CATALOG,
        "schema":  SCHEMA,
        "table_prefix": TABLE_PREFIX,
    },
    "query_engine": {
        "solver": "KeyValueStoreSolver",
        "data_type": "RAW",
        "solver_config": {
            "channel_mapping": {
                "join_keys": [{"mapping_col": "source_channel", "metrics_col": "channel_name"}]
            }
        },
    },
    "measurement_dimensions": ["container_id", "vehicle_key", "start_ts", "stop_ts"],
}

report = Report(
    name="key_value_store_demo",
    spark=spark,
    workspace_client=WorkspaceClient(),
    config=config,
)
db = report.get_db()

## Channel mapping & priority

The `channel_mapping` table maps physical channels to logical **aliases**. Two rows
point at the `vehicle_speed` alias:

| channel_alias | source_channel | priority | source_unit | target_unit |
|---|---|---|---|---|
| vehicle_speed | Vehicle Speed Sensor | 1 | km/h | m/s |
| vehicle_speed | veh_spd_ms | 2 | m/s | m/s |

Container 1 has **both** physical channels, so the alias resolves to the **lowest
priority number** (`Vehicle Speed Sensor`, priority 1) and converts km/h → m/s.
Containers 2 and 3 only have `Vehicle Speed Sensor`. Either way the alias yields one
consistent **m/s** signal.

In [ ]:
display(spark.read.table(f"{pfx}_channel_mapping"))
display(spark.read.table(f"{pfx}_unit_conversion"))

## Select by alias (automatic unit conversion)

`channel_with_alias` resolves the logical name and applies unit conversion.
Conversion only happens on **aliased** selectors — a direct `channel()` read returns
the raw stored values (km/h).

We build a histogram of `vehicle_speed` in **m/s** and, for contrast, the same
physical channel read directly in raw **km/h**. The `engine_speed` alias
(RPM → RPM, an identity mapping) is gated by an event to show event-scoped stats.

> Container filtering in this reduced model uses `MetricSelector` on wide
> `container_metrics` columns, e.g. `db.query.where(MetricSelector("vehicle_key") ==
> "Seat_Leon")` — **not** `TagSelector`, which needs a `container_tags` table.

In [ ]:
# Aliased selection -> resolved via channel_mapping; values converted to m/s.
veh_speed_ms = db.query.channel_with_alias(channel_alias="vehicle_speed")
eng_speed = db.query.channel_with_alias(channel_alias="engine_speed")  # RPM -> RPM (identity)

# Event expressed in CONVERTED units: ~50 km/h == 13.9 m/s.
fast = BasicEvent(name="fast", expr=veh_speed_ms > 13.9, desc="Vehicle speed above ~50 km/h (in m/s)")
report.add_event(fast)

page = Page(page_number=1)

# (1) Aliased histogram in m/s (conversion applied).
page.add_aggregation(
    HistogramDuration(
        name="speed_distribution_ms",
        base_expr=veh_speed_ms,
        bins=[x / 10 for x in range(0, 650, 50)],  # 0..60 m/s
        channel_name="vehicle_speed",
        bins_unit="m/s",
        values_unit="s",
    )
)

# (2) Direct read of the same physical channel in raw km/h (no conversion).
veh_speed_raw = db.query.channel(channel_name="Vehicle Speed Sensor")
page.add_aggregation(
    HistogramDuration(
        name="speed_distribution_kmh",
        base_expr=veh_speed_raw,
        bins=[float(x) for x in range(0, 240, 20)],  # 0..220 km/h
        channel_name="Vehicle Speed Sensor",
        bins_unit="km/h",
        values_unit="s",
    )
)

# (3) Aliased engine speed (RPM -> RPM identity), gated by the 'fast' event.
page.add_aggregation(
    HistogramDuration(
        name="engine_speed_when_fast",
        base_expr=eng_speed,
        bins=[float(x) for x in range(0, 5001, 500)],
        event=fast,
        channel_name="engine_speed",
        bins_unit="RPM",
        values_unit="s",
    )
)

report.add_page(page)
report.determine_report()
report.persist_results()

## Inspect the results

Join the histogram fact table to its dimension to label each histogram by name.
The `speed_distribution_ms` histogram occupies roughly **0.278×** the axis range of
`speed_distribution_kmh` — visual proof the alias applied the km/h → m/s conversion.

In [ ]:
import pyspark.sql.functions as F

fact = spark.read.table(f"{pfx}_histogram_fact")
dim = spark.read.table(f"{pfx}_histogram_dimension").select("visual_id", "name", "bins_unit")

display(
    fact.join(dim, on="visual_id")
    .groupBy("name", "bins_unit", "lower_bound", "upper_bound", "bin_name")
    .agg(F.sum("hist_value").alias("duration"))
    .orderBy("name", "lower_bound")
)